In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')


from sklearn.feature_selection import SelectKBest, f_regression

def selectkbest(indep_X, dep_Y, n, score_func=f_regression):
    test = SelectKBest(score_func=score_func, k=n)
    fit1 = test.fit(indep_X, dep_Y)
    selectk_features = fit1.transform(indep_X)
    return selectk_features


#def selectkbest(indep_X,dep_Y,n):
        #test = SelectKBest(score_func=chi2, k=n)
        #fit1= test.fit(indep_X,dep_Y)
        # summarize scores       
        #selectk_features = fit1.transform(indep_X)
        #return selectk_features
    
def split_scalar(indep_X,dep_Y):
        X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
        sc = StandardScaler()
        X_train = sc.fit_transform(X_train)
        X_test = sc.transform(X_test)    
        return X_train, X_test, y_train, y_test
    
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
 
def Linear(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.linear_model import LinearRegression
        regressor = LinearRegression()
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2   
    
def svm_linear(X_train,y_train,X_test):
        
        from sklearn.svm import SVR
        param_grid = {'kernel':['rbf','poly','sigmoid','linear'],
                    'C':[10,100,1000,2000,3000],'gamma':['auto','scale']}
        grid = GridSearchCV(SVR(), param_grid, refit = True, verbose = 3,n_jobs=-1)
                
        grid.fit(X_train, y_train)
        re=grid.cv_results_
        grid_predictions = grid.predict(X_test)
        r2=r2_score(y_test,grid_predictions)
        
        #r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
    
def svm_NL(X_train,y_train,X_test):
                
        from sklearn.svm import SVR
        regressor = SVR(kernel = 'rbf')
        regressor.fit(X_train, y_train)
        r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     

def Decision(X_train,y_train,X_test):
        
        from sklearn.tree import DecisionTreeRegressor
        param_grid = {'criterion':['mse','mae','friedman_mse'],'max_features': ['auto','sqrt','log2'],'splitter':['best','random']}
        grid = GridSearchCV(DecisionTreeRegressor(), param_grid, refit = True, verbose= 3,n_jobs=-1)
        # fitting the model for grid search
        grid.fit(X_train, y_train)
        re=grid.cv_results_
        grid_predictions = grid.predict(X_test)
        r2=r2_score(y_test,grid_predictions)
        #r2=r2_prediction(regressor,X_test,y_test)
        return  r2  
     

def random(X_train,y_train,X_test):       
        # Fitting K-NN to the Training set
        from sklearn.ensemble import RandomForestRegressor
        param_grid = {'criterion':['friedman_mse','squared_error','absolute_error','poisson'],'max_features': ['auto','sqrt','log2'],'n_estimators':[10,100]}
        grid = GridSearchCV(RandomForestRegressor(),param_grid, refit = True, verbose= 3,n_jobs=-1)
        grid.fit(X_train, y_train)
        re=grid.cv_results_
        grid_predictions = grid.predict(x_test)
        r2=r2_score(y_test,grid_predictions)
        #r2=r2_prediction(regressor,X_test,y_test)
        return  r2 
    
    
def selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf): 
    
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Linear','SVMl','SVMnl','Decision','Random'
                                                                                     ])

    for number,idex in enumerate(dataframe.index):
        
        dataframe['Linear'][idex]=acclin[number]       
        dataframe['SVMl'][idex]=accsvml[number]
        dataframe['SVMnl'][idex]=accsvmnl[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe
    
dataset=pd.read_csv("data/climate_change_impact_on_agriculture_2024.csv",index_col=None)

df=dataset

df = pd.get_dummies(df, drop_first=True)

indep_X=df.iloc[:,[1,14]].values
dep_Y=df['Economic_Impact_Million_USD']

indep_X += abs(indep_X.min())  # input X convert from negative to non negative

kbest = selectkbest(indep_X, dep_Y, 5, score_func=f_regression)
#kbest=selectkbest(indep_X,dep_Y,5)      

acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]

X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y)  
for i in kbest:   
    r2_lin=Linear(X_train,y_train,X_test)
    acclin.append(r2_lin)
    
    r2_sl=svm_linear(X_train,y_train,X_test)    
    accsvml.append(r2_sl)
    
    r2_NL=svm_NL(X_train,y_train,X_test)
    accsvmnl.append(r2_NL)
    
    r2_d=Decision(X_train,y_train,X_test)
    accdes.append(r2_d)
    
    r2_r=random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf)



Fitting 5 folds for each of 40 candidates, totalling 200 fits


In [ ]:
result